# PDB Graph Store usage exemple

## Installing dependences

In [1]:
!sudo apt update > /dev/null; sudo apt install -y gcc > /dev/null





debconf: delaying package configuration, since apt-utils is not installed


In [2]:
!pip install -r ./../r.txt > /dev/null

## Importing graphein modules

In [1]:
from graphein.protein.config import ProteinGraphConfig
from graphein.protein.graphs import construct_graph
from graphein.protein.utils import download_pdb

/opt/conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Importing graph store modules

In [2]:
from pkg.PDBGraphStore import PDBGraphStore

In [3]:
from pkg.operations import remove_graph_from_store
from pkg.operations import split_graph_store
from pkg.operations import merge_graph_stores

In [4]:
import pkg.edge_functions_Model as edgeModel

## Importing system modules

In [2]:
import traceback
import os
import time

In [3]:
print(time.time())

1790373409.5759168


## Reading dataset toy.txt at ./../data/

In [6]:
def read_dataset(general_data_path, dataset_txt_name, file_mode='r'):
    pdb_codes = list()
    with open(f"{general_data_path}/{dataset_txt_name}", file_mode) as file:
        for line in file:
            if line[0] != '#':
                pdb_codes.append(line.strip().upper())

    return pdb_codes

pdb_codes = read_dataset("./../data", "ligand_PLP.txt", 'r')
pdb_codes

['1B54',
 '1B8G',
 '1BJ4',
 '1C7G',
 '1F2D',
 '1FC4',
 '1HKV',
 '1I2K',
 '1JNW',
 '1JS3',
 '1KL7',
 '1LK9',
 '1LKC',
 '1M54',
 '1N31',
 '1O61',
 '1ORD',
 '1RFU',
 '1T3I',
 '1TDJ',
 '1UIM',
 '1V72',
 '1WYU',
 '1XRS',
 '1YGP',
 '2ABJ',
 '2AQ6',
 '2CFT',
 '2CTZ',
 '2DGM',
 '2E7I',
 '2FYF',
 '2HZP',
 '2JIS',
 '2UZP',
 '2Z67',
 '2ZY4',
 '3ANV',
 '3BB8',
 '3E77',
 '3F9T',
 '3HQT',
 '3I16',
 '3IF2',
 '3IHJ',
 '3KOW',
 '3LY1',
 '3MEB',
 '3N29',
 '3N2O',
 '3O05',
 '3PC2',
 '3SS7',
 '4BF5',
 '4CMD',
 '4H6D',
 '4M2J',
 '5DDS',
 '5HSJ',
 '5HXX',
 '5IJG',
 '5W71',
 '5X03']

## Defining the edge functions

In [7]:
def define_graphein_edge_funcs():
   return [edgeModel.edge_functions_dict[f] for f in ["delaunay", "aromatic", "aromatic_sulphur"]]

edge_funcs = define_graphein_edge_funcs()

edge_funcs

[<function graphein.protein.edges.distance.add_delaunay_triangulation(G: 'nx.Graph', allowable_nodes: 'Optional[List[str]]' = None)>,
 <function graphein.protein.edges.distance.add_aromatic_interactions(G: 'nx.Graph', pdb_df: 'Optional[pd.DataFrame]' = None)>,
 <function graphein.protein.edges.distance.add_aromatic_sulphur_interactions(G: 'nx.Graph', rgroup_df: 'Optional[pd.DataFrame]' = None)>]

## Defining graph configuration (with edge functions and node granularity)

In [8]:
def define_configuration(edge_construction_funcs):
    return {
        "granularity": "CA",
        "edge_construction_functions": edge_construction_funcs
    }

graph_config = ProteinGraphConfig(**define_configuration(edge_funcs))

graph_config

ProteinGraphConfig(granularity='CA', keep_hets=[], insertions=True, alt_locs='max_occupancy', pdb_dir=None, verbose=False, exclude_waters=True, deprotonate=False, protein_df_processing_functions=None, edge_construction_functions=[<function add_delaunay_triangulation at 0x7f076765b240>, <function add_aromatic_interactions at 0x7f076765ac00>, <function add_aromatic_sulphur_interactions at 0x7f076765aca0>], node_metadata_functions=[<function meiler_embedding at 0x7f0767665800>], edge_metadata_functions=None, graph_metadata_functions=None, get_contacts_config=None, dssp_config=None)

In [9]:
def get_pdb_file(pdb_data_path, pdb_code):
    pdb_code = pdb_code.lower()
    if os.path.exists(f"{pdb_data_path}/{pdb_code}.pdb"):
        print(f"Reading {pdb_code} from local directory")
        pdb_file = os.path.abspath(f"{pdb_data_path}/{pdb_code}.pdb")
    else:
        print(f"Downloading {pdb_code} from PDB")
        try:
            pdb_file = download_pdb(pdb_code, f"{pdb_data_path}/")
        except Exception as e:
            raise e

    if pdb_file == None:
        raise Exception("Error reading the pdb file")

    return pdb_file

In [10]:
def prepare_graph(pdb_data_path, pdb_code):
    protein_graphs = dict()

    try:
        pdb_file = get_pdb_file(pdb_data_path=pdb_data_path, pdb_code=pdb_code)
    except Exception as e:
        raise e

    graph = construct_graph(config=graph_config, path=pdb_file)
    graph.graph["pdb_code"] = pdb_code
    print(graph, "\n")

    config = graph.graph["config"]
    graph_pdb_code = graph.graph["pdb_code"]
    graph.graph.clear()
    graph.graph["config"] = config
    graph.graph["pdb_code"] = graph_pdb_code


    protein_graphs[pdb_code] = graph.copy()

    return protein_graphs

In [11]:
def build_pdb_store():
    n_nodes = 0
    n_edges = 0
    pdb_store = PDBGraphStore()

    for _, pdb_code in enumerate(pdb_codes.copy()):
        try:
            graph = prepare_graph("./../data/pdb_files", pdb_code)
            n_nodes += len(graph[pdb_code].nodes)
            n_edges += len(graph[pdb_code].edges)
        except Exception as e:
            msg = traceback.format_exc()
            print(msg)
            continue
        pdb_store.insert(graph)
    
    del graph

    return pdb_store, n_nodes, n_edges

In [12]:
pdb_graph_store, n_nodes, n_edges = build_pdb_store()
pdb_graph_store

/opt/conda/lib/python3.11/site-packages/rich/live.py:260: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

/opt/conda/lib/python3.11/site-packages/rich/live.py:260: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

/opt/conda/lib/python3.11/site-packages/rich/live.py:260: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

/opt/conda/lib/python3.11/site-packages/rich/live.py:260: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Reading 1b54 from local directory


...

/opt/conda/lib/python3.11/site-packages/rich/live.py:260: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

/opt/conda/lib/python3.11/site-packages/rich/live.py:260: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Reading 5x03 from local directory


/opt/conda/lib/python3.11/site-packages/rich/live.py:260: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Graph named '5x03' with 728 nodes and 5400 edges 



## Playing with pdb graph store object

In [13]:
print(len(pdb_graph_store.get_body_parts()['node_label_to_node_id']))
print(n_nodes)

35653
67803


In [14]:
print(len(pdb_graph_store.get_body_parts()['edge_label_to_edge_id']))
print(n_edges)

500704
510774


#### Getting the list of pdb codes in this object

In [15]:
print(pdb_graph_store.get_pdb_list())

dict_keys(['1b54', '1b8g', '1bj4', '1c7g', '1f2d', '1fc4', '1hkv', '1i2k', '1jnw', '1js3', '1kl7', '1lk9', '1lkc', '1m54', '1n31', '1o61', '1ord', '1rfu', '1t3i', '1tdj', '1uim', '1v72', '1wyu', '1xrs', '1ygp', '2abj', '2aq6', '2cft', '2ctz', '2dgm', '2e7i', '2fyf', '2hzp', '2jis', '2uzp', '2z67', '2zy4', '3anv', '3bb8', '3e77', '3f9t', '3hqt', '3i16', '3if2', '3ihj', '3kow', '3ly1', '3meb', '3n29', '3n2o', '3o05', '3pc2', '3ss7', '4bf5', '4cmd', '4h6d', '4m2j', '5dds', '5hsj', '5hxx', '5ijg', '5w71', '5x03'])


#### Extracting one pdb

In [16]:
g = pdb_graph_store.extract('1b54')

print(g.graph['pdb_code'])

1b54


#### Removing one pdb

In [17]:
pdb_graph_store = remove_graph_from_store(['1b54'], pdb_graph_store)

print(pdb_graph_store)

PDBGraphStore with 62 pdbs


In [18]:
print(pdb_graph_store.get_pdb_list())

dict_keys(['1b8g', '1bj4', '1c7g', '1f2d', '1fc4', '1hkv', '1i2k', '1jnw', '1js3', '1kl7', '1lk9', '1lkc', '1m54', '1n31', '1o61', '1ord', '1rfu', '1t3i', '1tdj', '1uim', '1v72', '1wyu', '1xrs', '1ygp', '2abj', '2aq6', '2cft', '2ctz', '2dgm', '2e7i', '2fyf', '2hzp', '2jis', '2uzp', '2z67', '2zy4', '3anv', '3bb8', '3e77', '3f9t', '3hqt', '3i16', '3if2', '3ihj', '3kow', '3ly1', '3meb', '3n29', '3n2o', '3o05', '3pc2', '3ss7', '4bf5', '4cmd', '4h6d', '4m2j', '5dds', '5hsj', '5hxx', '5ijg', '5w71', '5x03'])


#### Inserting one pdb

In [19]:
pdb_graph_store.insert({"1b54": g})

print(pdb_graph_store)

PDBGraphStore with 63 pdbs


In [20]:
print(pdb_graph_store.get_pdb_list())

dict_keys(['1b8g', '1bj4', '1c7g', '1f2d', '1fc4', '1hkv', '1i2k', '1jnw', '1js3', '1kl7', '1lk9', '1lkc', '1m54', '1n31', '1o61', '1ord', '1rfu', '1t3i', '1tdj', '1uim', '1v72', '1wyu', '1xrs', '1ygp', '2abj', '2aq6', '2cft', '2ctz', '2dgm', '2e7i', '2fyf', '2hzp', '2jis', '2uzp', '2z67', '2zy4', '3anv', '3bb8', '3e77', '3f9t', '3hqt', '3i16', '3if2', '3ihj', '3kow', '3ly1', '3meb', '3n29', '3n2o', '3o05', '3pc2', '3ss7', '4bf5', '4cmd', '4h6d', '4m2j', '5dds', '5hsj', '5hxx', '5ijg', '5w71', '5x03', '1b54'])


#### Spliting pdb store into 2 distincts stores

In [21]:
store1, store2 = split_graph_store(pdb_graph_store, ["1b54", "5x03"])

In [22]:
print(store1.get_pdb_list())
print(store2.get_pdb_list())

dict_keys(['5x03', '1b54'])
dict_keys(['1b8g', '1bj4', '1c7g', '1f2d', '1fc4', '1hkv', '1i2k', '1jnw', '1js3', '1kl7', '1lk9', '1lkc', '1m54', '1n31', '1o61', '1ord', '1rfu', '1t3i', '1tdj', '1uim', '1v72', '1wyu', '1xrs', '1ygp', '2abj', '2aq6', '2cft', '2ctz', '2dgm', '2e7i', '2fyf', '2hzp', '2jis', '2uzp', '2z67', '2zy4', '3anv', '3bb8', '3e77', '3f9t', '3hqt', '3i16', '3if2', '3ihj', '3kow', '3ly1', '3meb', '3n29', '3n2o', '3o05', '3pc2', '3ss7', '4bf5', '4cmd', '4h6d', '4m2j', '5dds', '5hsj', '5hxx', '5ijg', '5w71'])


In [23]:
merged = merge_graph_stores([store1, store2])

print(merged.get_pdb_list())
print(len(merged.get_pdb_list()))

{'config_CA_[]_True_max_occupancy_None_False_True_False_None_[<function add_delaunay_triangulation at 0x7f076765b240>, <function add_aromatic_interactions at 0x7f076765ac00>, <function add_aromatic_sulphur_interactions at 0x7f076765aca0>]_[<function meiler_embedding at 0x7f0767665800>]_None_None_None_None'}
dict_keys(['1b8g', '1bj4', '1c7g', '1f2d', '1fc4', '1hkv', '1i2k', '1jnw', '1js3', '1kl7', '1lk9', '1lkc', '1m54', '1n31', '1o61', '1ord', '1rfu', '1t3i', '1tdj', '1uim', '1v72', '1wyu', '1xrs', '1ygp', '2abj', '2aq6', '2cft', '2ctz', '2dgm', '2e7i', '2fyf', '2hzp', '2jis', '2uzp', '2z67', '2zy4', '3anv', '3bb8', '3e77', '3f9t', '3hqt', '3i16', '3if2', '3ihj', '3kow', '3ly1', '3meb', '3n29', '3n2o', '3o05', '3pc2', '3ss7', '4bf5', '4cmd', '4h6d', '4m2j', '5dds', '5hsj', '5hxx', '5ijg', '5w71', '1b54', '5x03'])
63
